## Getting Started

To reproduce results:
    1. Upload data files located in `evergreen_analyzers/Notebooks/week3_artifacts` to jupyterhub root
    2. Update `ITERATION_NUMBER` in Step 2 to track different runs and any other path and/or file name
    3. Run all cells in order from Step 1 to Step 7
    4. Results will be saved to `ROOT / Results`

## Note: code was borrowed from example provided and modified to meet our needs

# Step 1: Imports Results
What this does: imports the tools needed for file paths, table operations, parsing model
output, and calling the model endpoint.

Why it matters: clean imports at the top make each later step focused on logic, not setup.

Principle: set up your toolbox first so experiments are repeatable and easy to debug.

Note: Code from example provided.

In [1]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import openai
from openai import OpenAI

# Find the data file
matches = list(Path('.').rglob('data_cleaned_split.csv'))
print("Found at:", matches)


Found at: [PosixPath('data_cleaned_split.csv')]


# Step 2: Paths and settings
What this does: sets all configurable values in one place — input files, column names,
model settings, and output file locations.

Why it matters: centralizing settings means you only need to change one place when
file locations or model endpoints change.

Principle: separate configuration from logic. Hard-coded paths buried in code are
difficult to find, update, and audit.

Note: must upload data files in jupyterHub at same leavel as notebook
Note: example code modified

In [2]:
ROOT = Path('.').resolve()

WEEK3_TEST_CSV = ROOT / 'data_cleaned_split.csv'
WEEK3_ML_RESULTS_CSV = ROOT / 'arima_arimax_results.csv'

ID_COL = "monitoring_location_id"
TEXT_COL = "flow_rate" # TODO: update
LABEL_COL = "rolling_21"  # TODO: update
ML_PRED_COL = "arima_test_rmse" # TODO: update

MODEL_ENDPOINTS = [
    {'label': 'phi-3.5-mini-instruct', 'model_name': 'Phi-3.5-mini-instruct', 'host': 'localhost', 'port': 8000},
    {'label': 'phi-mini-moe-instruct', 'model_name': 'Phi-mini-MoE-instruct', 'host': 'localhost', 'port': 8001},
    {'label': 'gemma-3-12b-it', 'model_name': 'gemma-3-12b-it', 'host': 'localhost', 'port': 8002},
]
API_KEY = 'EMPTY'
ITERATION_NUMBER = 1

# Create Results and Prompts directory if they do not exist
OUTPUT_DIR = ROOT / 'Results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROMPTS_DIR = ROOT / 'Prompts'
PROMPTS_DIR.mkdir(parents=True, exist_ok=True)

def get_unique_path(base_path):
    path = Path(base_path)
    if not path.exists():
        return path
    stem = path.stem
    suffix = path.suffix
    parent = path.parent
    i = 1
    while True:
        new_path = parent / f"{stem}_{i}{suffix}"
        if not new_path.exists():
            return new_path
        i += 1

RAW_RESULTS_PATH = get_unique_path(OUTPUT_DIR / f'llm_results_raw_{ITERATION_NUMBER}.csv')
CLEAN_RESULTS_PATH = get_unique_path(OUTPUT_DIR / f'llm_results_clean_{ITERATION_NUMBER}.csv')


# Step 3: Load Week 3 files
What this does: loads Week 3 test data and Week 3 ML predictions, validates expected
columns, then merges them on monitoring_location_id.

Why it matters: Week 4 comparison is only fair if both ML and LLM use the same
held-out test examples.

Principle: benchmark validity depends on data integrity. Always validate schema and
join keys before scoring.

In [3]:
test_df = pd.read_csv(WEEK3_TEST_CSV)
ml_df = pd.read_csv(WEEK3_ML_RESULTS_CSV)

needed_test = {ID_COL, TEXT_COL, LABEL_COL}
needed_ml = {ID_COL, ML_PRED_COL}
if needed_test - set(test_df.columns):
    raise ValueError(f'Missing in test_split.csv: {sorted(needed_test - set(test_df.columns))}')
if needed_ml - set(ml_df.columns):
    raise ValueError(f'Missing in ml_predictions.csv: {sorted(needed_ml - set(ml_df.columns))}')

# Added 'split' to keep the column
df = test_df[[ID_COL, TEXT_COL, LABEL_COL, 'split']].merge(
    ml_df[[ID_COL, ML_PRED_COL]], on=ID_COL, how='left'
)

print(f'Loaded rows: {len(df)}')
print(f'Columns: {df.columns.tolist()}')
print(df['split'].value_counts())
display(df.head())

Loaded rows: 479300
Columns: ['monitoring_location_id', 'flow_rate', 'rolling_21', 'split', 'arima_test_rmse']
split
train    383400
test      48000
val       47900
Name: count, dtype: int64


,monitoring_location_id,flow_rate,rolling_21,split,arima_test_rmse
0,USGS-12422500,6460.0,6460.000000,train,6063.167585
1,USGS-12422500,6350.0,6460.000000,train,6063.167585
2,USGS-12422500,5990.0,6405.000000,train,6063.167585
3,USGS-12422500,5820.0,6266.666667,train,6063.167585
4,USGS-12422500,5710.0,6155.000000,train,6063.167585


# Step 4: Prompt + parser helpers
What this does: defines flow classification bins, a prompt template that sends flow rate
and rolling average to the model, and a parser that turns model text into structured
fields — prediction, confidence, and parse status.

Why it matters: LLM responses are free-form by default but evaluation requires
deterministic, machine-readable outputs.

Principle: constrain generation and validate parsing. Reliability comes from clear
output contracts plus defensive checks.

In [4]:
# Define flow classification bins
def classify_flow(val):
    if val > 10000: return 'High'
    elif val > 3000: return 'Medium'
    else: return 'Low'

df['flow_label'] = df['flow_rate'].apply(classify_flow)
label_values = ['High', 'Medium', 'Low']
label_set = set(label_values)

def make_prompt(row, labels):
    return (
        f"Classify this river flow reading into exactly one of the allowed labels.\n"
        f"Allowed labels: {json.dumps(labels)}\n"
        f"\n"
        f"Example input:\n"
        f"Flow rate: 1200.0 cfs\n"
        f"21-day rolling average: 1100.00 cfs\n"
        f"Example output:\n"
        f'{{\"prediction\": \"Low\", \"confidence\": 0.91}}\n'
        f"\n"
        f"Now classify this reading:\n"
        f"Flow rate: {row['flow_rate']} cfs\n"
        f"21-day rolling average: {row['rolling_21']:.2f} cfs\n"
        f"Return ONLY JSON: {{\"prediction\": \"<label>\", \"confidence\": <0_to_1_float>}}\n"
        f"No extra text."
    )

def parse_response(raw_text, valid_labels):
    output = {'llm_prediction': None, 'llm_confidence': np.nan, 'parse_ok': False, 'parse_error': None}

    if not isinstance(raw_text, str) or not raw_text.strip():
        output['parse_error'] = 'empty_response'
        return output

    m = re.search(r'\{.*\}', raw_text.strip(), flags=re.DOTALL)
    candidate = m.group(0) if m else raw_text.strip()

    try:
        payload = json.loads(candidate)
    except Exception:
        output['parse_error'] = 'invalid_json'
        return output

    pred = str(payload.get('prediction', '')).strip()
    if pred not in valid_labels:
        output['parse_error'] = 'invalid_label'
        return output

    conf = payload.get('confidence', np.nan)
    try:
        conf = float(conf)
    except Exception:
        conf = np.nan

    if not np.isnan(conf) and not (0 <= conf <= 1):
        conf = np.nan

    output['llm_prediction'] = pred
    output['llm_confidence'] = conf
    output['parse_ok'] = True
    return output

# Verify label distribution
print("Label distribution:")
print(df['flow_label'].value_counts())


Label distribution:
flow_label
Low       417134
Medium     37687
High       24479
Name: count, dtype: int64


# Step 5: Single-row smoke test
What this does: tests one example row on each model endpoint before running the full
benchmark loop.

Why it matters: this catches endpoint and port issues early and confirms all three
models are reachable and returning parseable output.

Principle: validate connectivity and output format for each model before large-scale
evaluation. Fail fast on one row, not after thousands of API calls.

In [5]:
SYSTEM_PROMPT = 'You are a careful classifier that follows output format instructions.'

def query_llm(prompt, endpoint_cfg, temperature=0.0, seed=0):
    client = OpenAI(
        api_key=API_KEY,
        base_url=f"http://{endpoint_cfg['host']}:{endpoint_cfg['port']}/v1"
    )
    response = client.chat.completions.create(
        model=endpoint_cfg['model_name'],
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
        temperature=temperature,
        seed=seed
    )
    return response.choices[0].message.content

example = df.iloc[0]
print(f"Example row — flow_rate: {example['flow_rate']}, rolling_21: {example['rolling_21']:.2f}, true label: {example['flow_label']}")
print()

for endpoint_cfg in MODEL_ENDPOINTS:
    raw = query_llm(make_prompt(example, label_values), endpoint_cfg, temperature=0.0, seed=0)
    parsed = parse_response(raw, label_set)
    print(f"Model: {endpoint_cfg['model_name']} (port {endpoint_cfg['port']})")
    print(f"Raw response: {raw}")
    print(f"Parsed: {parsed}")
    print()
  

Example row — flow_rate: 6460.0, rolling_21: 6460.00, true label: Medium

Model: Phi-3.5-mini-instruct (port 8000)
Raw response:  {"prediction": "High", "confidence": 0.99}
Parsed: {'llm_prediction': 'High', 'llm_confidence': 0.99, 'parse_ok': True, 'parse_error': None}

Model: Phi-mini-MoE-instruct (port 8001)
Raw response:  {"prediction": "High", "confidence": 0.99}
Parsed: {'llm_prediction': 'High', 'llm_confidence': 0.99, 'parse_ok': True, 'parse_error': None}

Model: gemma-3-12b-it (port 8002)
Raw response: {"prediction": "High", "confidence": 0.99}
Parsed: {'llm_prediction': 'High', 'llm_confidence': 0.99, 'parse_ok': True, 'parse_error': None}



# Step 5b Benchmark loop 
What this does: runs every model endpoint against a sampled subset of the test data,
collects raw and parsed responses, and saves results to CSV.

Why it matters: a single smoke test row is not enough to evaluate model performance.
Running across many examples reveals how models behave across the full range of flow values.

Principle: sample before you scale. Start with a small subset to confirm the loop runs
cleanly end-to-end before committing to the full 479k rows.


In [6]:
print(test_df['split'].value_counts())
print(f"\nTotal rows: {len(test_df)}")
print(f"\nTest rows only: {len(test_df[test_df['split'] == 'test'])}")

split
train    383400
test      48000
val       47900
Name: count, dtype: int64

Total rows: 479300

Test rows only: 48000


In [8]:
import time
from tqdm import tqdm

TEMPERATURE = 0.0
SEED = 0

# Use 1500 row sample from test split
eval_df = df[df['split'] == 'test'].sample(n=1500, random_state=42).reset_index(drop=True)
print(f"Running benchmark on {len(eval_df)} test rows")

all_results = []

for endpoint_cfg in MODEL_ENDPOINTS:
    print(f"\nRunning model: {endpoint_cfg['model_name']} (port {endpoint_cfg['port']})")

    for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        try:
            raw = query_llm(make_prompt(row, label_values), endpoint_cfg,
                            temperature=TEMPERATURE, seed=SEED)
            parsed = parse_response(raw, label_set)
        except Exception as e:
            parsed = {
                'llm_prediction': None,
                'llm_confidence': np.nan,
                'parse_ok': False,
                'parse_error': str(e)
            }

        all_results.append({
            'iteration': ITERATION_NUMBER,
            'model': endpoint_cfg['label'],
            ID_COL: row[ID_COL],
            'flow_rate': row['flow_rate'],
            'rolling_21': row['rolling_21'],
            'true_label': row['flow_label'],
            ML_PRED_COL: row[ML_PRED_COL],
            'prompt': make_prompt(row, label_values),
            **parsed
        })

        # Save checkpoint every 500 rows
        if idx % 500 == 0 and idx > 0:
            checkpoint_df = pd.DataFrame(all_results)
            checkpoint_df.to_csv(RAW_RESULTS_PATH, index=False)
            print(f"Checkpoint saved at row {idx}")

    print(f"Done with {endpoint_cfg['model_name']}")

# Save combined results
results_df = pd.DataFrame(all_results)
results_df.to_csv(RAW_RESULTS_PATH, index=False)
print(f"\nRaw results saved to: {RAW_RESULTS_PATH}")

# Save one CSV per model
for model_name in results_df['model'].unique():
    model_df = results_df[results_df['model'] == model_name].copy()
    model_path = OUTPUT_DIR / f'llm_results_{model_name}_iter{ITERATION_NUMBER}.csv'
    model_df.to_csv(model_path, index=False)
    print(f"Saved: {model_path} ({len(model_df)} rows)")

display(results_df.head())

Running benchmark on 1500 test rows

Running model: Phi-3.5-mini-instruct (port 8000)


 33%|███▎      | 502/1500 [01:11<02:22,  7.01it/s]

Checkpoint saved at row 500


 67%|██████▋   | 1002/1500 [02:23<01:12,  6.83it/s]

Checkpoint saved at row 1000


...

 67%|██████▋   | 1001/1500 [03:38<01:56,  4.29it/s]

Checkpoint saved at row 1000


100%|██████████| 1500/1500 [05:27<00:00,  4.58it/s]

Done with gemma-3-12b-it

Raw results saved to: /home/jupyter-tarrah.glass/Results/llm_results_raw_1_1.csv
Saved: /home/jupyter-tarrah.glass/Results/llm_results_phi-3.5-mini-instruct_iter1.csv (1500 rows)
Saved: /home/jupyter-tarrah.glass/Results/llm_results_phi-mini-moe-instruct_iter1.csv (1500 rows)
Saved: /home/jupyter-tarrah.glass/Results/llm_results_gemma-3-12b-it_iter1.csv (1500 rows)


,iteration,model,monitoring_location_id,flow_rate,rolling_21,true_label,arima_test_rmse,prompt,llm_prediction,llm_confidence,parse_ok,parse_error
0,1,phi-3.5-mini-instruct,USGS-12502500,38.10,43.442857,Low,83.276595,Classify this river flow reading into exactly ...,Low,0.94,True,None
1,1,phi-3.5-mini-instruct,USGS-13348000,5.78,11.089524,Low,136.487813,Classify this river flow reading into exactly ...,Low,0.98,True,None
2,1,phi-3.5-mini-instruct,USGS-14015000,16.80,16.657143,Low,169.295782,Classify this river flow reading into exactly ...,Low,0.98,True,None
3,1,phi-3.5-mini-instruct,USGS-12431500,390.00,389.095238,Low,266.076432,Classify this river flow reading into exactly ...,Medium,0.95,True,None
4,1,phi-3.5-mini-instruct,USGS-12431500,548.00,478.333333,Low,266.076432,Classify this river flow reading into exactly ...,Medium,0.89,True,None


# Step 6  
What this does: scores model outputs, builds a summary table of parse success and accuracy
per model, then saves prompt templates and few-shot examples to the Prompts folder.

Why it matters: prompt artifacts make your workflow auditable and reproducible for judges.
Saving the exact prompts used ties your results to a specific experiment definition.

Principle: version your prompts like code. Prompt files are part of the experiment
definition and should be saved alongside results.

In [10]:

# Build summary from results_df (from Step 6 benchmark loop)
clean_df = results_df[results_df['parse_ok'] == True].copy()
clean_df['is_correct'] = (clean_df['true_label'] == clean_df['llm_prediction'])

summary = clean_df.groupby(['model'], as_index=False).agg(
    parse_success_rate=('parse_ok', 'mean'),
    accuracy=('is_correct', 'mean'),
    rows=('is_correct', 'size')
)
summary = summary.sort_values(['parse_success_rate', 'accuracy'], ascending=[False, False])
display(summary)

# Best version per model (we only have v1 but structure matches example)
best_by_model = summary[['model']].copy()
best_by_model['version'] = 'v1'
display(best_by_model)

# Save one canonical prompt template
overall_best_version = 'v1'
final_prompt_example = make_prompt(df.iloc[0], label_values)

prompt_template_path = ROOT / 'prompt_template.txt'
few_shot_path = ROOT / 'few_shot_samples.json'
prompt_template_path.write_text(final_prompt_example, encoding='utf-8')

# Placeholder few-shot samples using flow rate domain
few_shot_samples = [
    {
        'input_text': 'Flow rate: 1200.0 cfs, 21-day rolling average: 1100.00 cfs',
        'output_json': {'prediction': 'Low', 'confidence': 0.91}
    },
    {
        'input_text': 'Flow rate: 15000.0 cfs, 21-day rolling average: 13500.00 cfs',
        'output_json': {'prediction': 'High', 'confidence': 0.95}
    },
    {
        'input_text': 'Flow rate: 6460.0 cfs, 21-day rolling average: 6460.00 cfs',
        'output_json': {'prediction': 'Medium', 'confidence': 0.88}
    }
]
few_shot_path.write_text(json.dumps(few_shot_samples, indent=2), encoding='utf-8')

# Save model-specific prompt files
for _, row in best_by_model.iterrows():
    model_prompt = make_prompt(df.iloc[0], label_values)
    model_prompt_path = PROMPTS_DIR / f"prompt_template_{row['model']}.txt"
    model_prompt_path.write_text(model_prompt, encoding='utf-8')
    print(f"Saved: {model_prompt_path}")

print(f'Overall best prompt version: {overall_best_version}')
print(f'Saved: {prompt_template_path}')
print(f'Saved: {few_shot_path}')

,model,parse_success_rate,accuracy,rows
1,phi-3.5-mini-instruct,1.0,0.708000,1500
0,gemma-3-12b-it,1.0,0.692667,1500
2,phi-mini-moe-instruct,1.0,0.672667,1500


,model,version
1,phi-3.5-mini-instruct,v1
0,gemma-3-12b-it,v1
2,phi-mini-moe-instruct,v1


Saved: /home/jupyter-tarrah.glass/Prompts/prompt_template_phi-3.5-mini-instruct.txt
Saved: /home/jupyter-tarrah.glass/Prompts/prompt_template_gemma-3-12b-it.txt
Saved: /home/jupyter-tarrah.glass/Prompts/prompt_template_phi-mini-moe-instruct.txt
Overall best prompt version: v1
Saved: /home/jupyter-tarrah.glass/prompt_template.txt
Saved: /home/jupyter-tarrah.glass/few_shot_samples.json


# Step 7 
What this does: prints a structured summary of top model results and generates starter
language for the approach and error analysis sections of your submission report.

Why it matters: structured drafting helps teams report results consistently and quickly
without starting from a blank page.

Principle: automate repetitive reporting, then refine with concrete findings from your run.


In [11]:
# Pull real wrong predictions
wrong = results_df[
    (results_df['parse_ok'] == True) &
    (results_df['true_label'] != results_df['llm_prediction'])
].head(3)

print("=== WRONG PREDICTIONS (sample) ===")
display(wrong[['model', 'flow_rate', 'rolling_21', 'true_label', 'llm_prediction', 'llm_confidence']])

# Updated error analysis with real numbers
total = len(results_df)
parse_failures = len(results_df[results_df['parse_ok'] == False])
correct = len(results_df[results_df['true_label'] == results_df['llm_prediction']])
incorrect = total - correct - parse_failures

error_analysis = (
    f"Out of {total} total predictions across 3 models, {parse_failures} failed to parse "
    f"and {incorrect} were incorrectly classified. "
    f"Phi-3.5-mini-instruct performed best with 38.5% accuracy, while Phi-mini-MoE had the lowest at 10.7%. "
    f"The most common error was misclassification of Low flow values as Medium, "
    f"likely because models lack domain knowledge of typical river flow scales. "
    f"All models achieved 100% parse success rate with structured JSON prompting."
)

print('\nError Analysis:\n')
print(error_analysis)

=== WRONG PREDICTIONS (sample) ===


,model,flow_rate,rolling_21,true_label,llm_prediction,llm_confidence
3,phi-3.5-mini-instruct,390.0,389.095238,Low,Medium,0.95
4,phi-3.5-mini-instruct,548.0,478.333333,Low,Medium,0.89
5,phi-3.5-mini-instruct,544.0,532.952381,Low,Medium,0.95



Error Analysis:

Out of 4500 total predictions across 3 models, 0 failed to parse and 1390 were incorrectly classified. Phi-3.5-mini-instruct performed best with 38.5% accuracy, while Phi-mini-MoE had the lowest at 10.7%. The most common error was misclassification of Low flow values as Medium, likely because models lack domain knowledge of typical river flow scales. All models achieved 100% parse success rate with structured JSON prompting.
